In [ ]:
!pip install rasterio pyarrow pyarrow-parquet

In [ ]:
import sys

# Install GDAL command-line utilities if not already installed
!apt-get update -qq && apt-get install -qq gdal-bin

try:
    import pyarrow.parquet
except ImportError:
    !pip install pyarrow --upgrade

print("GDAL and PyArrow dependencies checked/installed.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import glob
import os
import subprocess

def process_district_to_parquet(s2_pattern, target_tif_path, parquet_output_path, chunk_size=2048):
    # 1. Create the VRT
    s2_paths = glob.glob(s2_pattern)
    if not s2_paths:
        raise FileNotFoundError(f"No S2 tiles found for pattern: {s2_pattern}")

    vrt_path = "/content/temp_stitched.vrt"

    try:
        subprocess.run(['gdalbuildvrt', vrt_path] + s2_paths, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print("\nGDAL ERROR")
        print(e.stderr)
        raise

    # 2. Setup Schema for Parquet
    target_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    fields = [pa.field('target', pa.float32())]
    for b in target_bands:
        fields.append(pa.field(b, pa.float32()))
    fields.extend([pa.field('longitude', pa.float32()), pa.field('latitude', pa.float32())])
    schema = pa.schema(fields)

    # 3. Read the band names from the ORIGINAL file
    band_indices = {}
    with rasterio.open(s2_paths[0]) as first_tile:
        for i in range(1, first_tile.count + 1):
            desc = first_tile.descriptions[i - 1] or first_tile.tags(i).get('name', None)
            if desc and desc in target_bands:
                band_indices[desc] = i

    if not band_indices:
        raise ValueError("CRITICAL: Could not find band names in the original TIFFs!")

    # 4. Stream data in chunks
    with rasterio.open(vrt_path) as src, rasterio.open(target_tif_path) as tgt_src:
        with pq.ParquetWriter(parquet_output_path, schema, compression='snappy') as writer:
            for row_off in range(0, src.height, chunk_size):
                for col_off in range(0, src.width, chunk_size):

                    window = Window(col_off, row_off,
                                    min(chunk_size, src.width - col_off),
                                    min(chunk_size, src.height - row_off))

                    valid_mask = np.zeros(window.height * window.width, dtype=bool)
                    data_dict = {}

                    for band_name, idx in band_indices.items():
                        band_data = src.read(idx, window=window).flatten()
                        valid_mask |= (band_data > 0)
                        data_dict[band_name] = band_data

                    if not valid_mask.any():
                        continue

                    for b in target_bands:
                        data_dict[b] = data_dict[b][valid_mask].astype(np.float32)

                    target_data = tgt_src.read(1, window=window).flatten()
                    data_dict['target'] = target_data[valid_mask].astype(np.float32)

                    rows, cols = np.meshgrid(
                        np.arange(row_off, row_off + window.height),
                        np.arange(col_off, col_off + window.width),
                        indexing='ij'
                    )
                    valid_rows = rows.flatten()[valid_mask]
                    valid_cols = cols.flatten()[valid_mask]
                    xs, ys = rasterio.transform.xy(src.transform, valid_rows, valid_cols)

                    data_dict['longitude'] = np.array(xs, dtype=np.float32)
                    data_dict['latitude'] = np.array(ys, dtype=np.float32)

                    col_order = ['target'] + target_bands + ['longitude', 'latitude']
                    ordered_dict = {col: data_dict[col] for col in col_order}
                    table = pa.Table.from_pydict(ordered_dict, schema=schema)
                    writer.write_table(table)

    if os.path.exists(vrt_path):
        os.remove(vrt_path)


def batch_process_district(state_name, district_name, year="2024"):
    """Automates the pipeline across all 12 months for a given state and district."""

    months = {
        'January': '01', 'February': '02', 'March': '03', 'April': '04',
        'May': '05', 'June': '06', 'July': '07', 'August': '08',
        'September': '09', 'October': '10', 'November': '11', 'December': '12'
    }

    # Format names once to ensure consistency
    state_fmt = state_name
    district_fmt = district_name

    out_dir = f"/content/drive/MyDrive/bare_soil/parquet/{state_fmt}/{district_fmt}"
    os.makedirs(out_dir, exist_ok=True)

    for month_name, month_num in months.items():
        print(f"\n{'='*50}")
        print(f"🚀 Checking {state_fmt} -> {district_fmt} - {month_name} {year}")

        # --- PATH CONSTRUCTION ---
        base_dir_space = f"/content/drive/MyDrive/bare_soil/S2_Hybrid/{state_fmt}/{district_fmt}/{state_fmt}_{district_fmt} {month_name}_{year}"
        base_dir_slash = f"/content/drive/MyDrive/bare_soil/S2_Hybird/{state_fmt}/{district_fmt}/{state_fmt}_{district_fmt}/{month_name}_{year}"
        BASE_DIR = base_dir_slash if os.path.exists(base_dir_slash) else base_dir_space

        s2_pattern = os.path.join(BASE_DIR, f"{district_fmt}_{year}{month_num}_{month_name}_S2_AllBands*.tif")

        target_tif_file = os.path.join(BASE_DIR, f"{district_fmt}_{year}{month_num}_{month_name}_Hybrid.tif")
        parquet_file = os.path.join(out_dir, f"{district_fmt.lower()}_{year}{month_num}_{month_name.lower()}.parquet")

        # --- VALIDATION & EXECUTION ---
        if os.path.exists(parquet_file):
            print(f"SKIPPED: Parquet already exists -> {os.path.basename(parquet_file)}")
            continue

        if not glob.glob(s2_pattern):
            print(f"SKIPPED: No S2 tiles found matching pattern -> {os.path.basename(s2_pattern)}")
            continue

        if not os.path.exists(target_tif_file):
            print(f"SKIPPED: Hybrid mask not found -> {target_tif_file}")
            continue

        print(f"Data found! Building {os.path.basename(parquet_file)}...")
        try:
            process_district_to_parquet(s2_pattern, target_tif_file, parquet_file)
            size_mb = os.path.getsize(parquet_file) / 1e6
            print(f"SUCCESS! Saved {size_mb:.1f} MB to Drive.")
        except Exception as e:
            print(f"FAILED to process {month_name}: {e}")

if __name__ == "__main__":
    
    STATE_TO_PROCESS = "Punjab"
    DISTRICT_TO_PROCESS = "Amritsar"

    batch_process_district(STATE_TO_PROCESS, DISTRICT_TO_PROCESS)